Тестирование торговой стратегии

In [19]:
#!pip install backtesting

In [20]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from backtesting import Backtest, Strategy

In [21]:
%run prepare_data.ipynb

### Стратегия

Устанавливаем таргет через 1 час, обучаем модель, сохраняем в файл.

Если модель предсказывает, что актив будет расти - открываем ордер на покупку, если падать - шортим,
ставим стоп-лосс и тейк-профит, рассчитывая значения, учитывая текущую волатильность, по ATR.

Проверяем сделку через час, если тейк или стоп сработали, оцениваем результативность модели, если процент ошибок выше порога - начинаем процесс переобучения.

Если тейк или стоп не сработали, если модель предсказывает через час обратно открытой позиции, закрываем сделку.  
если в ту же сторону - продолжаем удерживать, двигаем стопы.

Для урежения сделок также принимаем решение, используя вероятность предсказания модели > 0.7 для лонга и  < 0.3 для шорта


In [22]:
class MainStrategy(Strategy):
	atr_period = 14
	sl_multiplier = 1.5
	tp_multiplier = 2.0

	def init(self):
		self.model = CatBoostClassifier()
		self.model.load_model('current_model.cbm')

		# Расчет волатильности
		high = pd.Series(self.data.High, index=self.data.index)
		low = pd.Series(self.data.Low, index=self.data.index)
		close = pd.Series(self.data.Close, index=self.data.index)

		# Расчет True Range (TR)
		tr = np.maximum(high - low,
						np.maximum(abs(high - close.shift(1)),
									abs(low - close.shift(1))))
		#Расчет ATR
		self.atr = tr.rolling(self.atr_period).mean()

		self.prepared_df, new_features = prepare_data(self.data.df.copy(), dividend_dates)

		scaler = StandardScaler()
		X_prepared = self.prepared_df[new_features]
		X_prepared_scaled = scaler.fit_transform(X_prepared)

		self.prepared_df['predictions'] = self.model.predict_proba(X_prepared_scaled)[:, 0]
		self.prepared_df.to_csv('prepared.csv')

		self.last_trade_time = None

	def next(self):
		inx=self.data.df.index[-1]

		if self.data.df.at[inx, 'Datetime'].minute != 0:
			return

		try:
			prediction = self.prepared_df.at[inx, 'predictions']
		except:
			print('Not found index', inx)
			return

		atr_value = self.atr[inx] 
		if atr_value == 0:
			return
		atr_value=3

		price = self.data.Close[-1]
		
		# Расчет уровней
		if prediction > 0.7:
			sl = price - self.sl_multiplier * atr_value  # Расчет стоп-лосс и тейк-проофита
			tp = price + self.tp_multiplier * atr_value
			if not self.position.is_long:
				self.buy(sl=sl, tp=tp)
				self.last_trade_time = self.data.index[-1]
		if prediction < 0.3:
			sl = price + self.sl_multiplier * atr_value
			tp = price - self.tp_multiplier * atr_value
			if not self.position.is_short:
				self.sell(sl=sl, tp=tp)
				self.last_trade_time = self.data.index[-1]

In [23]:
from backtesting import Backtest, Strategy
import joblib
import pandas as pd
import numpy as np

class MainStrategy_new(Strategy):
	sl_multiplier = 1.5   # Множители для стоп-лосса и тейк-профита
	tp_multiplier = 2.0  

	def init(self):
		self.model = CatBoostClassifier()
		self.model.load_model('current_model.cbm')  

		high = self.data.High
		low = self.data.Low
		close = self.data.Close

		# Расчёт True Range
		high_low = high - low
		high_close = np.abs(high - np.roll(close, 1))  # Сдвиг close на 1 назад
		low_close = np.abs(low - np.roll(close, 1))    # Сдвиг close на 1 назад

		# Объединяем три массива и берём максимум по оси 1
		true_range = np.maximum.reduce([high_low, high_close, low_close])

		# Инициализация первых значений как NaN (для корректности сдвига)
		true_range[0] = high_low[0]  # Первое значение не имеет предыдущего закрытия

		# Расчёт ATR с помощью экспоненциального сглаживания
		def ema(data, period):
			alpha = 2 / (period + 1)
			ema_values = np.zeros_like(data)
			ema_values[0] = data[0]  # Начальное значение
			for i in range(1, len(data)):
				ema_values[i] = alpha * data[i] + (1 - alpha) * ema_values[i-1]
			return ema_values

		self.atr = ema(true_range, 14)  # Период 14 для ATR
		
		self.last_trade_time = None
		self.min_interval = pd.Timedelta(minutes=60)

		# Список для хранения истории предсказаний (последние два предсказания)
		self.prediction_history = []

		self.processed_data, self.new_columns = prepare_data(self.data.df, dividend_dates)
		self.processed_data.set_index('Datetime', inplace=True)
		self.processed_data.to_csv('prepared.csv')

	def next(self):

		current_time = self.data.Datetime[-1]
		print (current_time)

		# Проверка, прошло ли достаточно времени с последней сделки
		if self.last_trade_time and (current_time - self.last_trade_time) < self.min_interval:
			return

		# Получение текущих признаков из синхронизированных данных
		try:
			current_features = self.processed_data.loc[current_time]
		except KeyError:
			print ("ErrorKey", current_time)
			return

		features_array = current_features.values.reshape(1, -1)

		# Предсказание модели и вероятность
		prediction = self.model.predict(features_array)[0]
		probability = self.model.predict_proba(features_array)[0][prediction]

		# Обновление истории предсказаний (храним только последние два предсказания)
		self.prediction_history.append(prediction)
		if len(self.prediction_history) > 2:
			self.prediction_history.pop(0)

		# Логика торговли: открываем позицию только если есть два совпадающих предсказания и вероятность > 70%
		if not self.position and len(self.prediction_history) == 2:
			if self.prediction_history[0] == self.prediction_history[1] and probability > 0.7:
				current_price = self.data.Close[-1]
				atr_value = self.atr[-1]
				
				if prediction == 1:  # Предсказан рост
					sl = current_price - atr_value * self.sl_multiplier
					tp = current_price + atr_value * self.tp_multiplier
					self.buy(sl=sl, tp=tp)
					self.last_trade_time = current_time
					
				elif prediction == 0:  # Предсказано падение
					sl = current_price + atr_value * self.sl_multiplier
					tp = current_price - atr_value * self.tp_multiplier
					self.sell(sl=sl, tp=tp)
					self.last_trade_time = current_time




In [24]:
df = pd.read_csv('sber_data.csv')
df['time'] = pd.to_datetime(df['time'])
start_date = df['time'].max() - pd.DateOffset(months=1)
df = df[df['time'] >= start_date]
df = df.reset_index(drop=True)


In [25]:
df.rename(columns={'time': 'Datetime', 'open':'Open','close':'Close', 'high':'High', 'low':'Low', 'volume':'Volume'}, inplace=True)

print(df.columns)

Index(['Open', 'High', 'Close', 'Low', 'Datetime', 'Volume'], dtype='object')


In [26]:
# Запуск бэктеста
bt = Backtest(df, MainStrategy, cash=10000, commission=0.002, exclusive_orders=True)
results = bt.run()

/tmp/ipykernel_390928/3837576796.py:2: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(df, MainStrategy, cash=10000, commission=0.002, exclusive_orders=True)


Not found index 29671


In [27]:
print(results)
bt.plot()

Start                                     0.0
End                                   29694.0
Duration                              29694.0
Exposure Time [%]                    89.90402
Equity Final [$]                       9083.9
Equity Peak [$]                     10091.538
Commissions [$]                        1010.1
Return [%]                             -9.161
Buy & Hold Return [%]                 2.28137
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                   -15.32177
Avg. Drawdown [%]                    -1.52892
Max. Drawdown Duration                28740.0
Avg. Drawdown Duration             2080.35714
# Trades                                 28.0
Win Rate [%]                         46.42857
Best Trade [%]                        2.47934
Worst Trade [%]                   

GridPlot(id='p1926', ...)